In [1]:
import os
import sys
import time
import zipfile
import subprocess

from google.colab import files
from google.colab.output import eval_js
from IPython.display import display, HTML

uploaded = files.upload()
zip_name = next(name for name in uploaded if name.endswith(".zip"))

ROOT = "/content/medical_record_integrity_security_edition_v2"

os.makedirs(ROOT, exist_ok=True)

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(ROOT)

PROJECT = os.path.join(ROOT, "medical_verifier_app")

assert os.path.isfile(os.path.join(PROJECT, "app.py"))
assert os.path.isfile(os.path.join(PROJECT, "requirements.txt"))

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "-r", os.path.join(PROJECT, "requirements.txt")],
    check=True
)

log = open("/content/medical_security_v2.log", "w")

server = subprocess.Popen(
    [sys.executable, "app.py"],
    cwd=PROJECT,
    stdout=log,
    stderr=subprocess.STDOUT
)

time.sleep(5)

if server.poll() is not None:
    print(open("/content/medical_security_v2.log").read()[-6000:])
else:
    url = eval_js("google.colab.kernel.proxyPort(5000)")
    print("Your website is ready:")
    display(HTML(f'<a href="{url}" target="_blank">Open your website</a>'))

Saving medical_record_integrity_security_edition_v2.zip to medical_record_integrity_security_edition_v2.zip
Your website is ready:


In [ ]:
import os
import signal
import subprocess
import time

# Stop only Flask processes running this project's app.py
result = subprocess.run(
    ["bash", "-lc",
     "ps -eo pid,args | grep '[p]ython.*app.py'"],
    capture_output=True, text=True
)
print("Existing app processes:")
print(result.stdout or "None found")

for line in result.stdout.splitlines():
    parts = line.strip().split(None, 1)
    if not parts:
        continue

    pid = int(parts[0])
    command = parts[1] if len(parts) > 1 else ""

    if "medical_verifier_app" in command and pid != os.getpid():
        try:
            os.kill(pid, signal.SIGTERM)
            print("Stopped old app process:", pid)
        except ProcessLookupError:
            pass

time.sleep(2)
print("\nNow rerun your original website-launch cell.")

Existing app processes:
    588 /usr/bin/python3 app.py


Now rerun your original website-launch cell.


In [ ]:
import os
import signal
import socket
import time

PID = 588

try:
    os.kill(PID, signal.SIGTERM)
    print("Sent stop signal to process", PID)
except ProcessLookupError:
    print("That process has already stopped.")
except PermissionError:
    print("Permission denied. Restart the Colab runtime instead.")

time.sleep(2)

sock = socket.socket()
result = sock.connect_ex(("127.0.0.1", 5000))
sock.close()

if result == 0:
    print("Port 5000 is still occupied. Restart the Colab runtime.")
else:
    print("Port 5000 appears free. Now run your website-launch cell once.")

Sent stop signal to process 588
Port 5000 appears free. Now run your website-launch cell once.
